- D-Fire RF-DETR-L | Kaggle | 2 GPU T4
  - Dataset: upload `D-Fire-train-ready.zip`; Kaggle Input cung cấp folder `D-Fire` đã sửa/audit local
  - Split khóa: `train`, `valid`, `test`; 800 px, 20 epoch, seed 20260707
  - Batch API 8 × accumulation 2; auto-batch target effective 16
  - Augmentation: trainer recipe mặc định RF-DETR; không custom override
  - LR recipe: native RF-DETR 1.8.3 (`step`, warmup 0); không ép cosine theo YOLO
  - Resume: tự quét working + Kaggle Input; Input cần `resume_protocol.json` cùng checkpoint
  - Output: `/kaggle/working/runs/dfire_rfdetr_large`


In [ ]:
!find /kaggle/input -maxdepth 6 -type d | head -100


In [ ]:
from pathlib import Path

SEED = 20260707
EPOCHS = 20
RESOLUTION = 800
INPUT_BASE = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'dfire_rfdetr_large'
MICRO_BATCH = 8
GRAD_ACCUM = 2
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
dataset_candidates = sorted({path.resolve() for path in INPUT_BASE.rglob('D-Fire') if (path / 'train' / 'images').is_dir() and (path / 'valid' / 'images').is_dir() and (path / 'test' / 'images').is_dir()})
assert len(dataset_candidates) == 1, f'Cần đúng 1 folder D-Fire trong Kaggle Input, thấy: {dataset_candidates}'
DATA_ROOT = dataset_candidates[0]
print({'data_root': str(DATA_ROOT), 'run_dir': str(RUN_DIR)})


In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[train]==1.8.3'], check=True)

import importlib.metadata
import torch

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
capabilities = [torch.cuda.get_device_capability(index) for index in range(torch.cuda.device_count())]
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'rfdetr': importlib.metadata.version('rfdetr'), 'devices': devices, 'capabilities': capabilities})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Kaggle 2 GPU T4, hiện có {devices}'
assert all(value >= (7, 0) for value in capabilities), f'GPU không được torch {torch.__version__} hỗ trợ: {capabilities}'


In [ ]:
from collections.abc import Mapping
import json
import os
import shutil

PROTOCOL = {'schema': 2, 'framework': 'rfdetr-1.8.3', 'model': 'RFDETRLarge', 'dataset': 'D-Fire-fixed-split-clip-edges-drop-degenerate', 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'micro_batch': MICRO_BATCH, 'grad_accum': GRAD_ACCUM, 'devices': 2, 'strategy': 'ddp_notebook', 'augmentation_policy': 'framework-default-no-user-overrides'}
protocol = RUN_DIR / 'resume_protocol.json'
existed = RUN_DIR.exists()
def valid_checkpoint(path, source):
    state = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(state, Mapping) or not isinstance(state.get('epoch'), int) or not isinstance(state.get('global_step'), int):
        raise ValueError('thiếu epoch/global_step')
    if not isinstance(state.get('state_dict'), Mapping) or not state['state_dict'] or not state.get('optimizer_states') or not state.get('lr_schedulers') or not isinstance(state.get('loops', {}).get('fit_loop'), Mapping):
        raise ValueError('không full-state')
    protocol_path = protocol if source == 'working' else next((parent / 'resume_protocol.json' for parent in (path.parent, *path.parents) if (parent / 'resume_protocol.json').is_file()), None)
    if protocol_path is None or json.loads(protocol_path.read_text(encoding='utf-8')) != PROTOCOL:
        raise ValueError('protocol không khớp')
    return {'path': path, 'source': source, 'epoch': state['epoch'], 'step': state['global_step']}
def scan_checkpoints(paths, source):
    accepted = []
    for path in paths:
        try:
            accepted.append(valid_checkpoint(path, source))
        except Exception as error:
            print('reject', source, path, error)
    return accepted
if existed and not protocol.is_file():
    raise RuntimeError('working thiếu protocol')
working_checkpoints = scan_checkpoints(RUN_DIR.glob('checkpoint_*.ckpt'), 'working')
if existed and not working_checkpoints:
    raise RuntimeError('working không có checkpoint hợp lệ')
input_checkpoints = scan_checkpoints(INPUT_BASE.rglob('checkpoint_*.ckpt'), 'input')
chosen = max(working_checkpoints + input_checkpoints, key=lambda item: (item['epoch'], item['step'], item['source'] == 'working'), default=None)
if chosen and chosen['source'] == 'input':
    destination = RUN_DIR / 'input_epoch_{:03d}_step_{:09d}.ckpt'.format(chosen['epoch'], chosen['step'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.tmp')
    shutil.copyfile(chosen['path'], temporary)
    os.replace(temporary, destination)
    resume_path = destination
else:
    resume_path = chosen['path'] if chosen else None
if not chosen and not existed:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    protocol.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding='utf-8')
run_complete = bool(chosen and chosen['epoch'] >= EPOCHS - 1)
print({'resume_source': chosen['source'] if chosen else 'fresh', 'resume_path': str(resume_path) if resume_path else None, 'complete': run_complete})


In [ ]:
from rfdetr import RFDETRLarge

if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}')
else:
    model = RFDETRLarge(resolution=RESOLUTION, num_classes=2)
    model.train(dataset_dir=str(DATA_ROOT), dataset_file='yolo', output_dir=str(RUN_DIR), epochs=EPOCHS, batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM, accelerator='gpu', devices=2, strategy='ddp_notebook', num_workers=2, checkpoint_interval=1, seed=SEED, early_stopping=False, resume=str(resume_path) if resume_path else None)


In [ ]:
checkpoints = sorted(RUN_DIR.glob('checkpoint_*.ckpt'))
weights = sorted(RUN_DIR.glob('*.pth'))
for path in checkpoints + weights:
    print(path, path.stat().st_size)
assert checkpoints
assert weights
